# 08 — GeoAnalytics: Spatial Hot Spot Analysis
**Powered by ArcGIS GeoAnalytics Engine in Microsoft Fabric**

Uses [Spark ArcGIS GeoAnalytics](https://learn.microsoft.com/en-us/fabric/data-engineering/spark-arcgis-geoanalytics)
to perform spatial analysis on bike stations — identifying statistically
significant clusters of underperforming and overperforming stations using the
**Getis-Ord Gi\*** hot spot statistic.

### What this produces

| Output Table | Lakehouse | Description |
|---|---|---|
| `geo_station_hotspots` | bicycles_gold | Hot/cold spot classification per station with Gi* z-scores |
| `geo_neighbourhood_zones` | bicycles_gold | Neighbourhood-level spatial risk zones |
| `geo_rebalancing_routes` | bicycles_gold | Proximity-based rebalancing suggestions (nearest critical → surplus) |

### Why this matters
Regular analytics tell you **which** stations are low. Spatial analytics tell you
**where clusters of low stations form** — and whether that pattern is statistically
significant or just random noise. Clustered problems need zone-level responses
(deploy a truck to the area), while isolated problems need station-level fixes.

### Prerequisites
1. Attach **bicycles_gold** as the default lakehouse
2. Add **bicycles_silver** as an additional lakehouse
3. Run **NB03 Silver** first so `silver_station_profile` exists
4. ArcGIS GeoAnalytics is pre-installed in Fabric Spark Runtime 1.3+

In [ ]:
# ============================================================
# CELL 1 — Imports & Configuration
# ============================================================

from pyspark.sql.functions import (
    col, lit, when, round as spark_round,
    count, avg as spark_avg, sum as spark_sum,
    min as spark_min, max as spark_max,
    current_timestamp, monotonically_increasing_id,
    row_number, sqrt, pow as spark_pow, acos, sin, cos, toRadians,
    expr, array, struct
)
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType, DoubleType, LongType, StringType
from datetime import datetime

# ── Lakehouse references ──────────────────────────────────────
SILVER_LAKEHOUSE = "bicycles_silver"
GOLD_LAKEHOUSE   = "bicycles_gold"

print(f"🌍 GeoAnalytics Notebook — {datetime.now()}")
print(f"   Source: {SILVER_LAKEHOUSE}")
print(f"   Target: {GOLD_LAKEHOUSE}")

In [ ]:
# ============================================================
# CELL 2 — Import ArcGIS GeoAnalytics
# ============================================================
# ArcGIS GeoAnalytics Engine is pre-installed in Fabric Spark
# Runtime 1.3+. If this fails, you're on an older runtime.

try:
    from geoanalytics_fabric.sql import functions as ST
    from geoanalytics_fabric import tools as geotools
    print("✅ ArcGIS GeoAnalytics Engine loaded successfully")
except ImportError as e:
    print(f"❌ ArcGIS GeoAnalytics not available: {e}")
    print("   Ensure you're on Fabric Spark Runtime 1.3+")
    print("   Falling back to Haversine-based analysis...")
    ST = None
    geotools = None

In [ ]:
# ============================================================
# CELL 3 — Read Station Data from Silver
# ============================================================

# ── Optimize bronze table (small-file compaction) ─────────────
# ── OPTIMIZE silver (small-file compaction) ─────────────────
try:
    spark.sql(f"OPTIMIZE {SILVER_LAKEHOUSE}.silver_station_profile")
    print("✅ Silver station profile OPTIMIZE complete")
except Exception as opt_err:
    print(f"⚠ OPTIMIZE skipped: {opt_err}")

try:
    df_stations = spark.sql(f"""
        SELECT 
            BikepointID   AS station_id,
            Street         AS street_address,
            Neighbourhood  AS neighbourhood,
            Latitude       AS latitude,
            Longitude      AS longitude,
            total_docks,
            utilization_pct,
            rebalance_priority,
            station_size,
            availability_status AS current_status
        FROM {SILVER_LAKEHOUSE}.silver_station_profile
    """)
    source = "silver_station_profile"
except Exception as e1:
    print(f"⚠ silver_station_profile not found: {e1}")
    print("  Falling back to bronze snapshot...")
    # Fallback: Build from Bronze directly — try dbo first (schema-enabled)
    df_raw = None
    for _fqn in ["bikerental_bronze_raw.dbo.bikeraw_tb", "bikerental_bronze_raw.bikeraw_tb"]:
        try:
            df_raw = spark.sql(f"SELECT * FROM {_fqn}")
            print(f"  ✓ Bronze fallback via: {_fqn}")
            break
        except Exception:
            pass
    if df_raw is None:
        raise RuntimeError("Could not read bikeraw_tb from bikerental_bronze_raw")
    
    df_stations = (
        df_raw
        .withColumn("latitude",  col("Latitude").cast(DoubleType()))
        .withColumn("longitude", col("Longitude").cast(DoubleType()))
        .withColumn("total_docks", col("No_Bikes").cast(IntegerType()) + col("No_Empty_Docks").cast(IntegerType()))
        .withColumn("utilization_pct", 
            when(col("total_docks") > 0,
                 spark_round(col("No_Bikes").cast(DoubleType()) / col("total_docks"), 4))
            .otherwise(lit(0.0)))
        .withColumnRenamed("BikepointID", "station_id")
        .withColumnRenamed("Street", "street_address")
        .withColumnRenamed("Neighbourhood", "neighbourhood")
        .withColumn("rebalance_priority", lit(0.5))
        .withColumn("station_size", lit("medium"))
        .withColumn("current_status", lit("Active"))
        .select("station_id", "street_address", "neighbourhood",
                "latitude", "longitude", "total_docks", "utilization_pct",
                "rebalance_priority", "station_size", "current_status")
    )
    source = "bronze_fallback"

ROW_COUNT_RAW = df_stations.count()
print(f"📥 Loaded {ROW_COUNT_RAW:,} rows from {source}")

# ── Deduplicate to latest snapshot per station ────────────────
# Bronze fallback may contain the full streaming history (many rows  
# per station). We only need one row per station for spatial analysis.
df_stations = df_stations.dropDuplicates(["station_id"])
ROW_COUNT = df_stations.count()
print(f"✅ Deduplicated to {ROW_COUNT:,} unique stations")
print(f"   Columns: {df_stations.columns}")
df_stations.show(5, truncate=False)

## Phase 1: Hot Spot Analysis (Getis-Ord Gi*)

The **Getis-Ord Gi\*** statistic identifies spatial clusters where station
utilization values are significantly higher or lower than expected by chance.

- **Hot spots** (Gi_Bin = +1 to +3): Clusters of HIGH utilization → bikes concentrated here
- **Cold spots** (Gi_Bin = -1 to -3): Clusters of LOW utilization → stations running empty
- **Not significant** (Gi_Bin = 0): Utilization is randomly distributed

Confidence levels: ±1 = 90%, ±2 = 95%, ±3 = 99%

In [ ]:
# ============================================================
# CELL 4 — Create Point Geometries & Run Hot Spot Analysis
# ============================================================

arcgis_succeeded = False
if ST is not None:
    try:
        # ── Create spatial points from lat/long ───────────────────
        df_geo = df_stations.withColumn(
            "geometry",
            ST.point(col("longitude"), col("latitude"), srid=4326)
        )
        
        print(f"✅ Created point geometries for {df_geo.count()} stations")
        
        # ── Run Getis-Ord Gi* Hot Spot Analysis ───────────────
        from geoanalytics_fabric.tools import FindHotSpots
        
        hotspots_result = FindHotSpots() \
            .setInputFeatures(df_geo) \
            .setAnalysisField("utilization_pct") \
            .run()
        
        df_hotspots = (
            hotspots_result
            .withColumn("hotspot_label", 
                when(col("Gi_Bin") >= 3,  "🔴 Hot Spot (99%%)")
                .when(col("Gi_Bin") == 2, "🟠 Hot Spot (95%%)")
                .when(col("Gi_Bin") == 1, "🟡 Hot Spot (90%%)")
                .when(col("Gi_Bin") <= -3, "🔵 Cold Spot (99%%)")
                .when(col("Gi_Bin") == -2, "🟣 Cold Spot (95%%)")
                .when(col("Gi_Bin") == -1, "⚪ Cold Spot (90%%)")
                .otherwise("⚫ Not Significant"))
            .withColumn("zone_action",
                when(col("Gi_Bin") >= 2,  "REDISTRIBUTE — bikes over-concentrated")
                .when(col("Gi_Bin") <= -2, "DEPLOY — bike desert, add bikes")
                .otherwise("MONITOR"))
        )
        
        arcgis_succeeded = True
        print("✅ Hot Spot Analysis complete (ArcGIS Gi*)")
    except Exception as arcgis_err:
        print(f"⚠ ArcGIS runtime error: {arcgis_err}")
        print("  Falling back to z-score spatial clustering...")

if not arcgis_succeeded:
    # ── Fallback: Z-Score based spatial clustering ────────────
    # When ArcGIS isn't available or fails at runtime,
    # use neighbourhood-level z-scores as a proxy for spatial clustering
    print("⚠ Using z-score fallback (ArcGIS not available)")
    
    from pyspark.sql.functions import stddev as spark_stddev, mean as spark_mean
    
    # Calculate mean and stddev of utilization
    stats = df_stations.select(
        spark_avg("utilization_pct").alias("global_mean"),
        spark_stddev("utilization_pct").alias("global_std")
    ).collect()[0]
    
    global_mean = stats["global_mean"]
    global_std  = stats["global_std"]
    
    # Neighbourhood-level aggregation
    df_neighbourhood_util = (
        df_stations
        .groupBy("neighbourhood")
        .agg(
            spark_avg("utilization_pct").alias("avg_utilization"),
            count("*").alias("station_count"),
            spark_avg("latitude").alias("centroid_lat"),
            spark_avg("longitude").alias("centroid_lon")
        )
    )
    
    # Z-score per station relative to global mean
    df_hotspots = (
        df_stations
        .withColumn("z_score", 
            spark_round((col("utilization_pct") - lit(global_mean)) / lit(global_std), 3))
        .withColumn("Gi_Bin",
            when(col("z_score") >= 2.576, lit(3))     # 99%% confidence
            .when(col("z_score") >= 1.96, lit(2))      # 95%%
            .when(col("z_score") >= 1.645, lit(1))     # 90%%
            .when(col("z_score") <= -2.576, lit(-3))
            .when(col("z_score") <= -1.96, lit(-2))
            .when(col("z_score") <= -1.645, lit(-1))
            .otherwise(lit(0)))
        .withColumn("hotspot_label",
            when(col("Gi_Bin") >= 3,  "🔴 Hot Spot (99%%)")
            .when(col("Gi_Bin") == 2, "🟠 Hot Spot (95%%)")
            .when(col("Gi_Bin") == 1, "🟡 Hot Spot (90%%)")
            .when(col("Gi_Bin") <= -3, "🔵 Cold Spot (99%%)")
            .when(col("Gi_Bin") == -2, "🟣 Cold Spot (95%%)")
            .when(col("Gi_Bin") == -1, "⚪ Cold Spot (90%%)")
            .otherwise("⚫ Not Significant"))
        .withColumn("zone_action",
            when(col("Gi_Bin") >= 2,  "REDISTRIBUTE — bikes over-concentrated")
            .when(col("Gi_Bin") <= -2, "DEPLOY — bike desert, add bikes")
            .otherwise("MONITOR"))
    )
    
    print("✅ Z-score spatial clustering complete")

# ── Preview results ───────────────────────────────────────────────
print(f"\n📊 Hot Spot Distribution:")
df_hotspots.groupBy("hotspot_label", "zone_action").count().orderBy("count", ascending=False).show(truncate=False)
df_hotspots.select("station_id", "street_address", "neighbourhood", 
                    "utilization_pct", "Gi_Bin", "hotspot_label", "zone_action").show(10, truncate=False)


## Phase 2: Neighbourhood Risk Zones

Aggregates hot spot results to the neighbourhood level to identify
which zones are **systematically** underperforming or overperforming.

In [ ]:
# ============================================================
# CELL 5 — Neighbourhood-Level Spatial Risk Zones
# ============================================================

df_zones = (
    df_hotspots
    .groupBy("neighbourhood")
    .agg(
        count("*").alias("station_count"),
        spark_avg("utilization_pct").alias("avg_utilization"),
        spark_avg("Gi_Bin").alias("avg_gi_score"),
        spark_sum(when(col("Gi_Bin") >= 2, 1).otherwise(0)).alias("hot_spot_count"),
        spark_sum(when(col("Gi_Bin") <= -2, 1).otherwise(0)).alias("cold_spot_count"),
        spark_sum(when(col("Gi_Bin") == 0, 1).otherwise(0)).alias("normal_count"),
        spark_avg("latitude").alias("centroid_lat"),
        spark_avg("longitude").alias("centroid_lon")
    )
    .withColumn("zone_risk",
        when(col("cold_spot_count") >= 3, "🔴 HIGH RISK — Bike Desert")
        .when(col("hot_spot_count") >= 3, "🟠 OVERSTOCK — Redistribution Needed")
        .when((col("cold_spot_count") >= 1) | (col("hot_spot_count") >= 1), "🟡 MODERATE — Monitor Closely")
        .otherwise("🟢 HEALTHY"))
    .withColumn("recommended_action",
        when(col("zone_risk").contains("HIGH RISK"), 
             "Deploy additional bikes, add stations, or incentivize riders to return here")
        .when(col("zone_risk").contains("OVERSTOCK"),
             "Redistribute bikes to neighbouring cold spots")
        .when(col("zone_risk").contains("MODERATE"),
             "Increase monitoring frequency, pre-position rebalancing trucks nearby")
        .otherwise("Maintain current operations"))
    .withColumn("avg_utilization", spark_round(col("avg_utilization"), 4))
    .withColumn("avg_gi_score", spark_round(col("avg_gi_score"), 2))
    .orderBy("avg_gi_score")
)

print("📊 Neighbourhood Risk Zones:")
df_zones.show(25, truncate=False)

## Phase 3: Proximity-Based Rebalancing Routes

For each **cold spot** (empty) station, find the nearest **hot spot** (full) station.
This creates natural rebalancing pairs: move bikes FROM the surplus TO the deficit.

Uses the **Haversine formula** to calculate distance between stations.
If ArcGIS is available, uses `ST.distance()` for geodesic accuracy.

In [ ]:
# ============================================================
# CELL 6 — Rebalancing Route Pairs (Nearest Surplus → Deficit)
# ============================================================

# Separate hot and cold spots
df_surplus  = df_hotspots.filter(col("Gi_Bin") >= 2).select(
    col("station_id").alias("surplus_station"),
    col("street_address").alias("surplus_street"),
    col("neighbourhood").alias("surplus_neighbourhood"),
    col("latitude").alias("surplus_lat"),
    col("longitude").alias("surplus_lon"),
    col("utilization_pct").alias("surplus_util")
)

df_deficit  = df_hotspots.filter(col("Gi_Bin") <= -2).select(
    col("station_id").alias("deficit_station"),
    col("street_address").alias("deficit_street"),
    col("neighbourhood").alias("deficit_neighbourhood"),
    col("latitude").alias("deficit_lat"),
    col("longitude").alias("deficit_lon"),
    col("utilization_pct").alias("deficit_util")
)

surplus_count = df_surplus.count()
deficit_count = df_deficit.count()
print(f"Surplus stations (Gi_Bin >= 2): {surplus_count}")
print(f"Deficit stations (Gi_Bin <= -2): {deficit_count}")

if surplus_count > 0 and deficit_count > 0:
    # Cross join all pairs (small dataset, ~195 stations max)
    df_pairs = df_deficit.crossJoin(df_surplus)
    
    # Haversine distance in km
    df_pairs = df_pairs.withColumn(
        "distance_km",
        spark_round(
            lit(6371) * acos(
                sin(toRadians(col("deficit_lat"))) * sin(toRadians(col("surplus_lat"))) +
                cos(toRadians(col("deficit_lat"))) * cos(toRadians(col("surplus_lat"))) *
                cos(toRadians(col("surplus_lon") - col("deficit_lon")))
            ),
            2
        )
    )
    
    # For each deficit station, find nearest surplus station
    w = Window.partitionBy("deficit_station").orderBy("distance_km")
    
    df_routes = (
        df_pairs
        .withColumn("rank", row_number().over(w))
        .filter(col("rank") == 1)
        .drop("rank")
        .withColumn("bikes_to_move", 
            spark_round((col("surplus_util") - lit(0.5)) * lit(20), 0).cast(IntegerType()))
        .withColumn("route_priority",
            when(col("distance_km") < 1, "🟢 SHORT (<1 km)")
            .when(col("distance_km") < 3, "🟡 MEDIUM (1-3 km)")
            .otherwise("🔴 LONG (>3 km)"))
        .select(
            "deficit_station", "deficit_street", "deficit_neighbourhood", "deficit_util",
            "surplus_station", "surplus_street", "surplus_neighbourhood", "surplus_util",
            "distance_km", "bikes_to_move", "route_priority"
        )
        .orderBy("distance_km")
    )
    
    print(f"\n🚛 Rebalancing Routes ({df_routes.count()} pairs):")
    df_routes.show(20, truncate=False)
else:
    print("⚠ No surplus/deficit pairs found — fleet may be well-balanced")
    # Create empty DataFrame with schema
    from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType
    schema = StructType([
        StructField("deficit_station", StringType()),
        StructField("deficit_street", StringType()),
        StructField("deficit_neighbourhood", StringType()),
        StructField("deficit_util", DoubleType()),
        StructField("surplus_station", StringType()),
        StructField("surplus_street", StringType()),
        StructField("surplus_neighbourhood", StringType()),
        StructField("surplus_util", DoubleType()),
        StructField("distance_km", DoubleType()),
        StructField("bikes_to_move", IntegerType()),
        StructField("route_priority", StringType()),
    ])
    df_routes = spark.createDataFrame([], schema)

## Phase 4: Save to Gold Lakehouse

All three geo-analytical outputs are persisted as Delta tables in the Gold lakehouse,
making them available to:
- **RTI Dashboard** — KQL queries via shortcuts
- **Power BI** — Direct Lake semantic model
- **Data Agent** — Natural language queries

In [ ]:
# ============================================================
# CELL 7 — Write to Gold Lakehouse
# ============================================================

OUTPUT_TABLES = {
    "geo_station_hotspots": df_hotspots.select(
        "station_id", "street_address", "neighbourhood",
        "latitude", "longitude", "total_docks",
        "utilization_pct", "rebalance_priority",
        "Gi_Bin", "hotspot_label", "zone_action"
    ),
    "geo_neighbourhood_zones": df_zones,
    "geo_rebalancing_routes": df_routes,
}

for table_name, df in OUTPUT_TABLES.items():
    target = f"{GOLD_LAKEHOUSE}.{table_name}"
    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(target)
    row_count = spark.sql(f"SELECT COUNT(*) as cnt FROM {target}").collect()[0]["cnt"]
    print(f"✅ {target} — {row_count:,} rows")

print(f"\n🎉 GeoAnalytics pipeline complete — {datetime.now()}")

## Phase 5: Summary Visualization

Quick summary of the spatial analysis for validation before moving to the dashboard.

In [ ]:
# ============================================================
# CELL 8 — Summary Statistics
# ============================================================

print("=" * 60)
print("  🌍 GEOANALYTICS SUMMARY")
print("=" * 60)

# Hot Spot Distribution
print("\n📍 Station Hot Spot Classification:")
df_hotspots.groupBy("hotspot_label").count().orderBy("count", ascending=False).show(truncate=False)

# Neighbourhood Risk
print("\n🏘️ Neighbourhood Risk Summary:")
df_zones.select("neighbourhood", "station_count", "avg_utilization", 
                "hot_spot_count", "cold_spot_count", "zone_risk").show(25, truncate=False)

# Route Stats
route_count = df_routes.count()
if route_count > 0:
    avg_distance = df_routes.select(spark_avg("distance_km")).collect()[0][0]
    total_bikes = df_routes.select(spark_sum("bikes_to_move")).collect()[0][0]
    print(f"\n🚛 Rebalancing Routes:")
    print(f"   Pairs identified : {route_count}")
    print(f"   Avg distance     : {avg_distance:.1f} km")
    print(f"   Total bikes to move: {total_bikes}")
else:
    print("\n🚛 No rebalancing routes needed — fleet is balanced!")

print("\n" + "=" * 60)
print("  ✅ Ready for dashboard integration")
print("     Tables available in: bicycles_gold")
print("=" * 60)

## KQL Queries for RTI Dashboard

After creating a **shortcut** from your Eventhouse to the `bicycles_gold` lakehouse
(or querying via the Lakehouse SQL endpoint), use these queries in your dashboard:

### Station Hot Spot Map
```kql
geo_station_hotspots
| project station_id, street_address, neighbourhood, latitude, longitude,
    utilization_pct, Gi_Bin, hotspot_label, zone_action
| order by Gi_Bin desc
```
Visual type: **Map** (latitude + longitude columns auto-detected, color by Gi_Bin)

### Neighbourhood Risk Zones
```kql
geo_neighbourhood_zones
| project neighbourhood, station_count, avg_utilization, avg_gi_score,
    hot_spot_count, cold_spot_count, zone_risk, recommended_action
| order by avg_gi_score asc
```
Visual type: **Table** (conditional formatting on zone_risk)

### Rebalancing Routes
```kql
geo_rebalancing_routes
| project deficit_street, deficit_neighbourhood, surplus_street,
    surplus_neighbourhood, distance_km, bikes_to_move, route_priority
| order by distance_km asc
```
Visual type: **Table** (conditional formatting on route_priority)